In [4]:
import numpy as np
import pandas as pd

In [2]:
year = "2021"
df = pd.read_csv(f"../Data/Procesed data/Cleaned data/{year}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)

In [3]:
df = df.drop("void_grubs",axis=1)

KeyError: "['void_grubs'] not found in axis"

In [ ]:
df.head()

# Quitando nulos

In [39]:
#Por desgracia la instrucción df.isnull().sum() no imprime todas las columnas porque son muchas. Esto es una vista similar
def enseña_nulos(my_df):
    nulls = my_df.isnull().sum()
    columns = my_df.columns
    for i,null in enumerate(nulls):
        print(f"{columns[i]:<20}:{null}")
    print(my_df.shape)

In [40]:
enseña_nulos(df)

gameid              :0
side                :0
p1_name             :107
p1_champion         :0
p1_kills            :0
p1_deaths           :0
p1_pentakill        :2110
p1_assist           :0
p1_dpm              :12
p1_cspm             :0
p1_vspm             :2138
p1_firstblood       :12
p1_firstbloodassist :2148
p1_goldat10         :2150
p1_goldat15         :2152
p1_xpat10           :2150
p1_xpat15           :2152
p2_name             :128
p2_champion         :0
p2_kills            :0
p2_deaths           :0
p2_pentakill        :2110
p2_assist           :0
p2_dpm              :12
p2_cspm             :0
p2_vspm             :2138
p2_firstblood       :12
p2_firstbloodassist :2148
p2_goldat10         :2150
p2_goldat15         :2152
p2_xpat10           :2150
p2_xpat15           :2152
p3_name             :113
p3_champion         :0
p3_kills            :0
p3_deaths           :0
p3_pentakill        :2110
p3_assist           :0
p3_dpm              :12
p3_cspm             :0
p3_vspm             :213

Los nombres tanto de jugadores como de equipos no necesitan no tener nulos porque no se van a usar para el entrenamiento sino para seleccionar unas u otras filas. 

Las pentakill en nulo se sustituirán por 0. Así como dragones, barons, heralds, firstbloodassist. Tiene sentido que los valores sean null. En una partida un jugador o equipo puede no haber matado dragones, barons o heralds y pueden no haber tenido ninguna asistencia.

Quedan como nulls goldat y xpat 10 y 15 respectivamente. Se rellenarán los que se puedan con la media de sus columnas de acuerdo a sus jugadores y sus equipos. Los que queden después, se rellenarán con la media de la columna. Se hará lo mismo con px_vspm.

Los nulos en las columnas px_dpm y px_firstblood son un problema.

In [38]:
for i in range(1,6):
    print(f"Nulos dpm y firstblood de p{i}:", len(df[df[f"p{i}_dpm"].isna() | df[f"p{i}_firstblood"].isna()]))

Nulos dpm y firstblood de p1: 12
Nulos dpm y firstblood de p2: 12
Nulos dpm y firstblood de p3: 12
Nulos dpm y firstblood de p4: 12
Nulos dpm y firstblood de p5: 12


Parece ser que los mismos nulos que se encuentran en una columna también sucede en la otra, solo en 12 y para todos los jugadores al mismo tiempo por igual. Por lo tanto, no sería muy dañíno si simplemente se eliminan.

In [56]:
for i in range(1,6):
    df[f"p{i}_pentakill"] = df[f"p{i}_pentakill"].fillna(0)
    df[f"p{i}_dpm"] = df[f"p{i}_dpm"].fillna(
        df.groupby(f"p{i}_name")[f"p{i}_dpm"].transform("mean")
    )
    df[f"p{i}_vspm"] = df[f"p{i}_vspm"].fillna(
        df.groupby(f"p{i}_name")[f"p{i}_vspm"].transform("mean")
    )
    df[f"p{i}_vspm"] = df[f"p{i}_vspm"].fillna(
        df[f"p{i}_vspm"].mean()
    )
    
    df[f"p{i}_goldat10"] = df[f"p{i}_goldat10"].fillna(
        df.groupby(f"p{i}_name")[f"p{i}_goldat10"].transform("mean")
    )
    df[f"p{i}_goldat10"] = df[f"p{i}_goldat10"].fillna(
        df[f"p{i}_goldat10"].mean()
    )
    
    df[f"p{i}_goldat15"] = df[f"p{i}_goldat15"].fillna(
        df.groupby(f"p{i}_name")[f"p{i}_goldat15"].transform("mean")
    )
    df[f"p{i}_goldat15"] = df[f"p{i}_goldat15"].fillna(
        df[f"p{i}_goldat15"].mean()
    )
    
    df[f"p{i}_xpat10"] = df[f"p{i}_xpat10"].fillna(
        df.groupby(f"p{i}_name")[f"p{i}_xpat10"].transform("mean")
    )
    df[f"p{i}_xpat10"] = df[f"p{i}_xpat10"].fillna(
        df[f"p{i}_xpat10"].mean()
    )
    
    df[f"p{i}_xpat15"] = df[f"p{i}_xpat15"].fillna(
        df.groupby(f"p{i}_name")[f"p{i}_xpat15"].transform("mean")
    )
    df[f"p{i}_xpat15"] = df[f"p{i}_xpat15"].fillna(
        df[f"p{i}_xpat15"].mean()
    )
    
    df[f"p{i}_firstbloodassist"] = df[f"p{i}_firstbloodassist"].fillna(0)
    
df["dragons"] = df["dragons"].fillna(0)
df["barons"] = df["barons"].fillna(0)
df["heralds"] = df["heralds"].fillna(0)

df["goldat10"] = df["goldat10"].fillna(
    df.groupby("team")["goldat10"].transform("mean")
)
df["goldat10"] = df["goldat10"].fillna(
    df["goldat10"].mean()
)
    
df["goldat15"] = df["goldat15"].fillna(
    df.groupby("team")["goldat15"].transform("mean")
)
df["goldat15"] = df["goldat15"].fillna(
    df["goldat15"].mean()
)
    
df["xpat10"] = df["xpat10"].fillna(
    df.groupby("team")["xpat10"].transform("mean")
)
df["xpat10"] = df["xpat10"].fillna(
    df["xpat10"].mean()
)
    
df["xpat15"] = df["xpat15"].fillna(
    df.groupby("team")["xpat15"].transform("mean")
)
df["xpat15"] = df["xpat15"].fillna(
    df["xpat15"].mean()
)

In [57]:
enseña_nulos(df)

gameid              :0
side                :0
p1_name             :107
p1_champion         :0
p1_kills            :0
p1_deaths           :0
p1_pentakill        :0
p1_assist           :0
p1_dpm              :0
p1_cspm             :0
p1_vspm             :0
p1_firstblood       :12
p1_firstbloodassist :0
p1_goldat10         :0
p1_goldat15         :0
p1_xpat10           :0
p1_xpat15           :0
p2_name             :128
p2_champion         :0
p2_kills            :0
p2_deaths           :0
p2_pentakill        :0
p2_assist           :0
p2_dpm              :0
p2_cspm             :0
p2_vspm             :0
p2_firstblood       :12
p2_firstbloodassist :0
p2_goldat10         :0
p2_goldat15         :0
p2_xpat10           :0
p2_xpat15           :0
p3_name             :113
p3_champion         :0
p3_kills            :0
p3_deaths           :0
p3_pentakill        :0
p3_assist           :0
p3_dpm              :0
p3_cspm             :0
p3_vspm             :0
p3_firstblood       :12
p3_firstbloodassist :0
p3

In [60]:
print(df[df["p1_firstblood"].isna()])
    

          gameid   side    p1_name p1_champion  p1_kills  p1_deaths  \
46     6918-9217   True   hulatang        Gnar         2          3   
47     6918-9217  False     TheShy       Vayne         1          5   
840    7087-9622   True  Xiaolaohu       Jayce         4          3   
841    7087-9622  False     lcberg        Gnar         2          3   
1008   7123-9746   True     Jotaro    Renekton         3          2   
1009   7123-9746  False     Sakura        Gnar         0          2   
1306   7311-9697   True       Cube       Jayce         3          0   
1307   7311-9697  False        New        Gnar         1          3   
2922  7754-10589   True      Aliez      Aatrox         4          1   
2923  7754-10589  False     Flying        Sett         0          3   
3134  7798-10738   True  Xiaolaohu        Gwen         7          1   
3135  7798-10738  False     lcberg        Sett         1          3   

      p1_pentakill  p1_assist      p1_dpm  p1_cspm  ...  \
46             0.

In [52]:
df["side"] = df["side"].map(lambda x: True if x =="Blue" else False)

In [53]:
df.head()

,gameid,side,p1_name,p1_champion,p1_kills,p1_deaths,p1_pentakill,p1_assist,p1_dpm,p1_cspm,...,team,dragons,barons,heralds,towers,result,goldat10,goldat15,xpat10,xpat15
0,6909-9183,True,369,Karma,0,0,0.0,2,539.8244,8.0780,...,Top Esports,2.0,0.0,0.0,3.0,0,16177.0,24815.0,19640.0,31121.0
1,6909-9183,False,Bin,Aatrox,1,0,0.0,6,230.0780,9.2488,...,Suning,2.0,1.0,0.0,6.0,1,15445.0,23864.0,19565.0,31228.0
2,6909-9184,True,369,Jax,0,7,0.0,5,346.8273,8.1928,...,Top Esports,3.0,2.0,0.0,6.0,0,16752.0,27355.0,20020.0,32158.0
3,6909-9184,False,Bin,Camille,11,3,0.0,12,688.4070,7.2289,...,Suning,2.0,0.0,0.0,7.0,1,15250.0,25210.0,18856.0,32578.0
4,6910-9189,True,New,Camille,0,3,0.0,1,268.1717,8.8394,...,Oh My God,1.0,0.0,0.0,3.0,0,15842.0,24131.0,18405.0,29284.0


# Los campeones deben ser números

La mejor opción sería convertir la columna a categoría. Ocurre aunque es poco probable que algún campeón no se encuentre en un año y bailen los números. Para ello se ha confeccionado un diccionario con los campeones de todos los años asignándoles un valor alfabeticamente. Solo restará añadir una columna champion_id en función del champion de cada jugador.

In [28]:
champ_df = pd.read_csv("../Data/Aux data/champion_mapping.csv")

cmp_dict = pd.Series(champ_df.champion_id.values, index=champ_df.champion).to_dict()

# Accessing the champion ID
print(cmp_dict["Akshan"])  # Output: 3

3


# Definido como función

In [47]:
def arregla_datos(my_df,champs,this_year):
    for i in range(1,6):
        my_df[f"p{i}_pentakill"] = df[f"p{i}_pentakill"].fillna(0)
        my_df[f"p{i}_dpm"] = my_df[f"p{i}_dpm"].fillna(
            my_df.groupby(f"p{i}_name")[f"p{i}_dpm"].transform("mean")
        )
        my_df[f"p{i}_vspm"] = my_df[f"p{i}_vspm"].fillna(
            my_df.groupby(f"p{i}_name")[f"p{i}_vspm"].transform("mean")
        )
        my_df[f"p{i}_vspm"] = my_df[f"p{i}_vspm"].fillna(
            my_df[f"p{i}_vspm"].mean()
        )
        
        my_df[f"p{i}_goldat10"] = my_df[f"p{i}_goldat10"].fillna(
            my_df.groupby(f"p{i}_name")[f"p{i}_goldat10"].transform("mean")
        )
        my_df[f"p{i}_goldat10"] = my_df[f"p{i}_goldat10"].fillna(
            my_df[f"p{i}_goldat10"].mean()
        )
        
        my_df[f"p{i}_goldat15"] = my_df[f"p{i}_goldat15"].fillna(
            my_df.groupby(f"p{i}_name")[f"p{i}_goldat15"].transform("mean")
        )
        my_df[f"p{i}_goldat15"] = my_df[f"p{i}_goldat15"].fillna(
            my_df[f"p{i}_goldat15"].mean()
        )
        
        my_df[f"p{i}_xpat10"] = my_df[f"p{i}_xpat10"].fillna(
            my_df.groupby(f"p{i}_name")[f"p{i}_xpat10"].transform("mean")
        )
        my_df[f"p{i}_xpat10"] = my_df[f"p{i}_xpat10"].fillna(
            my_df[f"p{i}_xpat10"].mean()
        )
        
        my_df[f"p{i}_xpat15"] = my_df[f"p{i}_xpat15"].fillna(
            my_df.groupby(f"p{i}_name")[f"p{i}_xpat15"].transform("mean")
        )
        my_df[f"p{i}_xpat15"] = my_df[f"p{i}_xpat15"].fillna(
            my_df[f"p{i}_xpat15"].mean()
        )
        
        my_df[f"p{i}_firstbloodassist"] = my_df[f"p{i}_firstbloodassist"].fillna(0)
        #my_df[f"p{i}_champion"] = df[f"p{i}_champion"].astype("category")
        #my_df[f"p{i}_champion_id"] = my_df[f"p{i}_champion"].map(champs)
        
        my_df.insert(4+(16*(i-1)),f"p{i}_champion_id",my_df[f"p{i}_champion"].map(champs))
        
    my_df["dragons"] = my_df["dragons"].fillna(0)
    my_df["barons"] = my_df["barons"].fillna(0)
    my_df["heralds"] = my_df["heralds"].fillna(0)
    
    my_df["goldat10"] = my_df["goldat10"].fillna(
        my_df.groupby("team")["goldat10"].transform("mean")
    )
    my_df["goldat10"] = my_df["goldat10"].fillna(
        my_df["goldat10"].mean()
    )
        
    my_df["goldat15"] = my_df["goldat15"].fillna(
        my_df.groupby("team")["goldat15"].transform("mean")
    )
    my_df["goldat15"] = my_df["goldat15"].fillna(
        my_df["goldat15"].mean()
    )
        
    my_df["xpat10"] = my_df["xpat10"].fillna(
        my_df.groupby("team")["xpat10"].transform("mean")
    )
    my_df["xpat10"] = my_df["xpat10"].fillna(
        my_df["xpat10"].mean()
    )
        
    my_df["xpat15"] = my_df["xpat15"].fillna(
        my_df.groupby("team")["xpat15"].transform("mean")
    )
    my_df["xpat15"] = my_df["xpat15"].fillna(
        my_df["xpat15"].mean()
    )
    my_df["side"] = my_df["side"].map(lambda x: True if x =="Blue" else False)

    my_df.to_csv(f"../Data/Procesed data/Re-formulated data/{this_year}_LoL_esports_match_data_from_OraclesElixir.csv", index=False)

In [ ]:
years = list(range(2014,2027))

for y in years:
    this_df =  pd.read_csv(f"../Data/Procesed data/Cleaned data/{y}_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)
    arregla_datos(this_df,cmp_dict,y)
    print(f"year {y}, done!")

year 2014, done!
year 2015, done!
year 2016, done!
year 2017, done!
year 2018, done!
year 2019, done!
year 2020, done!
year 2021, done!
year 2022, done!
year 2023, done!


In [48]:
this_df =  pd.read_csv(f"../Data/Procesed data/Cleaned data/2024_LoL_esports_match_data_from_OraclesElixir.csv",low_memory=False)
arregla_datos(this_df,cmp_dict,1)